Realizamos las importaciones necesarias:

In [2]:
import os
import csv
import ast
import re
import time
import nltk
import collections
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support, f1_score
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.base import clone
from joblib import parallel_backend
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments, 
    EarlyStoppingCallback, 
    set_seed, 
    default_data_collator
)

Fijamos las variables globales del cuaderno:

In [3]:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "clasificacion-de-papers/data"
MAX_LEN=512
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "dataset_test.csv")
TEST_FILE_LABELS = os.path.join(CORPUS_DIR, "etiquetas_test_sin_alias.csv")
EMB_FILE = "embeddings_specter2.npz"
SPECTER2 = "allenai/specter2_base"
SPECTER2_DIR = "specter2/no_duplicados"
SCIBERT = "allenai/scibert_scivocab_uncased"
SCIBERT_DIR = "scibert_final/model"
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios
set_seed(SEED)

Definimos funciones de carga de datos, tokenización, y cálculo de métricas y estadísticas:

In [4]:
# Función para cargar el corpus:
def load_corpus(filename, delimiter=',', duplicates=False, is_test=False):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Creamos un set para los títulos ya encontrados a fin de evitar duplicados:
        titles = set()

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            title = row[0].strip().lower()
            summary = row[1].strip().lower()
            keywords = row[2].strip().lower() if (is_test and len(row) > 2) else ""

            if not duplicates and not is_test:
                if title not in titles:
                    titles.add(title)
                else:
                    continue

            papers.append((title, summary, keywords))
            if not is_test:
                terms.append(ast.literal_eval(row[-1]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    probs = {}
    for label, freq in sorted(ct.items(), key=lambda item: item[1], reverse=True):
        prob = freq / len(texts)
        probs[label] = prob
        print(f"{label}: {freq} ({prob:.1%})", end="  ")
    print("\n")
    return probs


try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

In [5]:
# Definimos una función para ajustar los umbrales de decisión de un clasificador:
def adjust_thresholds(model, X, y, min_th=0.01, max_th=1, n_th=200, alg="rl"):
    if model is not None:
        # Obtenemos las probabilidades para cada clase que ha asociado el modelo a cada paper:
        scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)
    else:
        scores = X

    # Creamos un vector con tantos elementos como clases, inicializado a ceros:
    n_classes = len(mlb.classes_)
    optimal_thresholds = np.zeros(n_classes)

    # Definimos una lista de 'n_th' umbrales de decisión equiespaciados entre 'min_th' y 'max_th':
    thresholds = np.linspace(min_th, max_th, n_th)

    # Iteramos sobre cada clase:
    for c in range(n_classes):
        max_f1_c = -1
        best_th = 0

        # Obtenemos la columna de etiquetas y probabilidades para cada paper asociadas a la clase:
        y_c = y[:, c]
        scores_c = scores[:, c]

        # Iteramos sobre cada umbral:
        for th in thresholds:
            # Obtenemos las predicciones de etiquetas asociadas a ese umbral y calculamos su F1 individual:
            preds_c = (scores_c >= th).astype(int)
            f1_c = f1_score(y_c, preds_c, average="binary", zero_division=0)

            # Si el F1 mejora respecto al que tenía la clase ya, lo guardamos:
            if f1_c > max_f1_c:
                max_f1_c = f1_c
                best_th = th

        # En cada entrada 'i' de 'optimal_thresholds', almacenamos el mejor umbral encontrado para la clase 'c_i':
        optimal_thresholds[c] = best_th

    return optimal_thresholds

# Definimos asimismo una función para obtener las predicciones optimizadas dados los umbrales ajustados:
def predict_with_th(model, X, y, thresholds, verbose=False, alg="rl"):
    if model is not None:
        # Obtenemos las probabilidades originales:
        scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)
    else:
        scores = X

    # Actualizamos las predicciones con los umbrales de decisión que hemos calculado:
    preds = (scores >= thresholds).astype(int)

    # Buscamos los papers que se hayan quedado sin clase:
    zero_rows = np.where(preds.sum(axis=1) == 0)[0] # NumPy devuelve una tupla de 1 elemento, por eso lo extraemos

    # Iteramos sobre cada paper sin clase, obtenemos la clase de mayor probabilidad y se la asignamos:
    for r in zero_rows:
        top_class = np.argmax(scores[r])
        preds[r, top_class] = 1

    if verbose:
        # Consultamos las métricas derivadas de usarlos:
        print(compute_metrics(y, preds))

    return preds

In [6]:
class PaperDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

Carga de archivos:

In [7]:
df_train = pd.read_csv(TRAIN_FILE)
df_train['title_clean'] = df_train['titles'].astype(str).str.strip().str.lower()
df_train = df_train.drop_duplicates(subset=['title_clean']).reset_index(drop=True)

# Binarizamos las etiquetas originales de arXiv:
train_terms = [ast.literal_eval(terms) for terms in df_train['terms']]
mlb = MultiLabelBinarizer()
y_all = mlb.fit_transform(train_terms)
num_classes = len(mlb.classes_)

df_test = pd.read_csv(TEST_FILE)

In [9]:
_, labels = load_corpus(TEST_FILE_LABELS, delimiter=';')

SPECTER2

In [10]:
specter_tokenizer = AutoTokenizer.from_pretrained(SPECTER2)

# Unimos título y resumen usando el token separador nativo del modelo:
train_texts = [f"{t}{specter_tokenizer.sep_token}{s}" for t, s in zip(df_train['titles'], df_train['summaries'])]

# Definimos una partición de entrenamiento y otra de desarrollo:
X_train, X_dev, y_train, y_dev = train_test_split(train_texts, y_all, test_size=0.15, random_state=2627, shuffle=True)

test_texts = [f"{t}{specter_tokenizer.sep_token}{s}{specter_tokenizer.sep_token}{k}" for t, s, k in zip(df_test['titles'], df_test['summaries'], df_test["keywords"])]

test_dataset_specter = PaperDataset(test_texts, None, specter_tokenizer)

train_dataset_specter = PaperDataset(X_train, y_train, specter_tokenizer)
dev_dataset_specter = PaperDataset(X_dev, y_dev, specter_tokenizer)

specter_model = AutoModelForSequenceClassification.from_pretrained(SPECTER2_DIR)

# Crear el Trainer solo para inferencia/evaluación
specter_trainer = Trainer(
    model=specter_model,
    args=TrainingArguments(
        output_dir="./tmp_eval",
        per_device_eval_batch_size=32,
        report_to="none"
    ),
    data_collator=default_data_collator
)

print("Obteniendo predicciones del modelo SPECTER cargado...")
dev_pred_specter = specter_trainer.predict(dev_dataset_specter)
dev_probs_specter = 1 / (1 + np.exp(-dev_pred_specter.predictions))

test_pred_specter = specter_trainer.predict(test_dataset_specter)
test_probs_specter = 1 / (1 + np.exp(-test_pred_specter.predictions))

metrics_specter_alone = compute_metrics(y_dev, predict_with_th(dev_probs_specter, np.full(num_classes, 0.5)))
print(f"-> Aislado SciBERT (Fijo 0.5) | F1 Macro DEV: {metrics_specter_alone['f1']:.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 635.13it/s]
c:\Users\icata\anaconda3\envs\paperClassifier\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Obteniendo predicciones del modelo SPECTER cargado...


KeyboardInterrupt: 

SCIBERT

In [11]:
scibert_tokenizer = AutoTokenizer.from_pretrained(SCIBERT)

# Unimos título y resumen usando el token separador nativo del modelo:
train_texts = [f"{t}{scibert_tokenizer.sep_token}{s}" for t, s in zip(df_train['titles'], df_train['summaries'])]

# Definimos una partición de entrenamiento y otra de desarrollo:
X_train, X_dev, y_train, y_dev = train_test_split(train_texts, y_all, test_size=0.15, random_state=2627, shuffle=True)

test_texts = [f"{t}{scibert_tokenizer.sep_token}{s}{scibert_tokenizer.sep_token}{k}" for t, s, k in zip(df_test['titles'], df_test['summaries'], df_test["keywords"])]

test_dataset_scibert = PaperDataset(test_texts, None, scibert_tokenizer)

train_dataset_scibert = PaperDataset(X_train, y_train, scibert_tokenizer)
dev_dataset_scibert = PaperDataset(X_dev, y_dev, scibert_tokenizer)

scibert_model = AutoModelForSequenceClassification.from_pretrained(SCIBERT_DIR)

# Crear el Trainer solo para inferencia/evaluación
scibert_trainer = Trainer(
    model=scibert_model,
    args=TrainingArguments(
        output_dir="./tmp_eval",
        per_device_eval_batch_size=32,
        report_to="none"
    ),
    data_collator=default_data_collator
)

print("Obteniendo predicciones del modelo SCIBERT cargado...")
dev_pred_scibert = scibert_trainer.predict(dev_dataset_scibert)
dev_probs_scibert = 1 / (1 + np.exp(-dev_pred_scibert.predictions))

test_pred_scibert = scibert_trainer.predict(test_dataset_scibert)
test_probs_scibert = 1 / (1 + np.exp(-test_pred_scibert.predictions))

metrics_scibert_alone = compute_metrics(y_dev, predict_with_th(dev_probs_scibert, np.full(num_classes, 0.5)))
print(f"-> Aislado SciBERT (Fijo 0.5) | F1 Macro DEV: {metrics_scibert_alone['f1']:.4f}")

c:\Users\icata\anaconda3\envs\paperClassifier\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\icata\.cache\huggingface\hub\models--allenai--scibert_scivocab_uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 518.92it/s]


Obteniendo predicciones del modelo SCIBERT cargado...


KeyboardInterrupt: 

RL

In [ ]:
raw_all, y_all = load_corpus(TRAIN_FILE, duplicates=False)

raw_test, _ = load_corpus(TEST_FILE, delimiter=',', is_test=True)

mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
num_classes = len(mlb.classes_)

# Formatear textos
train_texts = [f"{t} {s}" for t, s, _ in raw_all]

X_train, X_dev_tok, y_train, y_dev = train_test_split(train_texts, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

test_texts = [f"{t} {s} {k}" for t, s, k in raw_test]

In [ ]:
vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=1)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_dev_tfidf = vectorizer.transform(X_dev)
X_test_tfidf = vectorizer.transform(train_texts)

lr_model = OneVsRestClassifier(LogisticRegression(solver='liblinear', C=1.0, class_weight="balanced", random_state=SEED)).fit(X_train_tfidf, y_train)
dev_probs_lr = lr_model.predict_proba(X_dev_tfidf)

X_train_full = vectorizer.transform(train_texts)

lr_model_full = OneVsRestClassifier(LogisticRegression(solver='liblinear', C=1.0, class_weight="balanced", random_state=SEED)).fit(X_train_full, y_bin) # se reentrena con todo
test_probs_lr = lr_model_full.predict_proba(X_test_tfidf)

metrics_lr_alone = compute_metrics(y_dev, predict_with_th(dev_probs_lr, np.full(num_classes, 0.5)))

ENSEMBLE SCIBERT + SPECTER 2

In [ ]:
# Combinación ponderada de probabilidades
for a in range(1, 11, 1):
    alpha = a/100

    dev_probs_ensemble = alpha * dev_probs_scibert + (1 - alpha) * dev_probs_specter
    test_probs_ensemble = alpha * test_probs_scibert + (1 - alpha) * test_probs_specter

    metrics_fixed = compute_metrics(y_dev, predict_with_th(dev_probs_ensemble, np.full(num_classes, 0.5)))

    opt_thresholds = adjust_thresholds(dev_probs_ensemble, y_dev)
    preds_dev_opt = predict_with_th(dev_probs_ensemble, opt_thresholds)
    metrics_opt = compute_metrics(y_dev, preds_dev_opt)


    print("\n" + "="*80)
    print(" RESULTADOS FINALES EN DEV (VALIDACIÓN)")
    print("="*80)
    print(f"1. SPECTER Aislado                              | F1 Macro: {metrics_specter_alone['f1']:.4f}")
    print(f"2. SciBERT Aislado                              | F1 Macro: {metrics_scibert_alone['f1']:.4f}")
    print(f"3. Ensamble                                     | F1 Macro: {metrics_fixed['f1']:.4f}")
    print(f"4. Ensamble + Umbrales Optimizados              | F1 Macro: {metrics_opt['f1']:.4f}")
    print("="*80)

    # Evaluación
    test_preds_fixed = predict_with_th(test_probs_ensemble, predict_with_th(dev_probs_ensemble, np.full(num_classes, 0.5)))
    metrics_final_fixed = compute_metrics(y_dev, test_preds_fixed)
    print(f"Ensamble                            | F1 Macro: {metrics_final_fixed['f1']:.4f}")

    test_preds_final = predict_with_th(test_probs_ensemble, opt_thresholds)
    metrics_final_th = compute_metrics(y_dev, test_preds_final)
    print(f"Ensamble + Umbrales Optimizados     | F1 Macro: {metrics_final_th['f1']:.4f}")

    print(f"F1 real fijo: {compute_metrics(labels, test_preds_fixed)}")
    print(f"F1 real umbrales: {compute_metrics(labels, test_preds_final)}")

ENSEMBLE SCIBERT + RL

In [ ]:
# Combinación ponderada de probabilidades
for a in range(1, 11, 1):
    alpha = a/100

    dev_probs_ensemble = alpha * dev_probs_scibert + (1 - alpha) * dev_probs_lr
    test_probs_ensemble = alpha * test_probs_scibert + (1 - alpha) * test_probs_lr

    metrics_fixed = compute_metrics(y_dev, predict_with_th(dev_probs_ensemble, np.full(num_classes, 0.5)))

    opt_thresholds = adjust_thresholds(dev_probs_ensemble, y_dev)
    preds_dev_opt = predict_with_th(dev_probs_ensemble, opt_thresholds)
    metrics_opt = compute_metrics(y_dev, preds_dev_opt)


    print("\n" + "="*80)
    print(" RESULTADOS FINALES EN DEV (VALIDACIÓN)")
    print("="*80)
    print(f"1. LR Aislado                                   | F1 Macro: {metrics_lr_alone['f1']:.4f}")
    print(f"2. SciBERT Aislado                              | F1 Macro: {metrics_scibert_alone['f1']:.4f}")
    print(f"3. Ensamble                                     | F1 Macro: {metrics_fixed['f1']:.4f}")
    print(f"4. Ensamble + Umbrales Optimizados              | F1 Macro: {metrics_opt['f1']:.4f}")
    print("="*80)

    # Evaluación
    test_preds_fixed = predict_with_th(test_probs_ensemble, predict_with_th(dev_probs_ensemble, np.full(num_classes, 0.5)))
    metrics_final_fixed = compute_metrics(y_dev, test_preds_fixed)
    print(f"Ensamble                            | F1 Macro: {metrics_final_fixed['f1']:.4f}")

    test_preds_final = predict_with_th(test_probs_ensemble, opt_thresholds)
    metrics_final_th = compute_metrics(y_dev, test_preds_final)
    print(f"Ensamble + Umbrales Optimizados     | F1 Macro: {metrics_final_th['f1']:.4f}")

    print(f"F1 real fijo: {compute_metrics(labels, test_preds_fixed)}")
    print(f"F1 real umbrales: {compute_metrics(labels, test_preds_final)}")